# Import e set-up iniziale


In [ ]:
import scipy.stats as stats
import time
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import networkx as nx
from scipy.stats import fisher_exact
from statsmodels.stats.multitest import multipletests
from pyvis.network import Network
import warnings
from itertools import combinations

# Ignoriamo i warning sui dtypes misti per letture veloci
warnings.filterwarnings('ignore', category=pd.errors.DtypeWarning)

# Setup grafica
sns.set_theme(style="whitegrid")
print("✅ Librerie caricate con successo. Ambiente pronto.")

In [ ]:
# ==========================================
# SELEZIONE MODALITA DI ANALISI
# ==========================================
# 1 = Solo Mutazioni (SNV)
# 2 = Solo CNA (Amplificazioni/Delezioni)
# 3 = Solo SV (Varianti Strutturali)


ANALYSIS_MODE = 1

# ANALISI KRAS


## Config


In [ ]:
# ==========================================
# CONFIGURAZIONE KRAS CENTRICA 
# ==========================================

# Soglie Statistiche per identificare partner
TARGET_GENE = 'KRAS'
P_VALUE_THRESH = 0.05   # Significatività statistica grezza
LOG2OR_THRESH = {
    "pancreas": 1.0,
    "lung": 1.5,
    "colon": 2.5
}

# Soglie per la Mutua Esclusività
P_VALUE_THRESH_ME = 0.05
# Vogliamo un Log2OR (-1.0 significa probabilità dimezzata di trovarli insieme)
LOG2OR_THRESH_ME = -1.0 

# Filtri Grafici (per Heatmap e Network)
MIN_COOCCURRENCE_NETWORK = {
    "pancreas": 4,
    "lung": 10,
    "colon": 40
}
TOP_GENES_HEATMAP = 20         # Quanti geni mostrare nella heatmap iniziale

# Percorsi Cartelle
COORTI = {
    "pancreas": "./kras_pancreas",
    "lung": "./kras_lung",
    "colon": "./kras_colon"
}

print(f"⚙️ Parametri impostati: {TARGET_GENE} con p-value < {P_VALUE_THRESH} e Log2OR > {LOG2OR_THRESH}")


## Matrici


### Star co-occurrence matrix, gene A always KRAS


In [ ]:
# ==========================================
# CELLA 4: GENERAZIONE DATI CON SELETTORE
# ==========================================
def generate_star_cooccurrence_data(path_cohort, cohort_name, output_dir, mode=ANALYSIS_MODE):
    # Mapping nomi per i file di output
    labels = {1: "MUT", 2: "CNA", 3: "SV"}
    tag = labels.get(mode, "CUSTOM")
    
    mut_file = f"{path_cohort}/F_data_mutations.txt"
    cna_file = f"{path_cohort}/F_data_cna.txt"
    sv_file = f"{path_cohort}/F_data_sv.csv"

    if not os.path.exists(mut_file): 
        print(f"[!] File base non trovato per {cohort_name}. Salto.")
        return None
        
    # DataFrame per raccogliere gli eventi selezionati
    events = pd.DataFrame(columns=['Sample_Id', 'Hugo_Symbol'])

    # --- A. LOGICA MUTAZIONI (Mode 1) ---
    if mode == 1:
        df_mut = pd.read_csv(mut_file, sep='\t')
        functional = ['Missense_Mutation', 'Nonsense_Mutation', 'Frame_Shift_Del', 
                      'Frame_Shift_Ins', 'In_Frame_Del', 'In_Frame_Ins', 'Splice_Site']
        df_mut_func = df_mut[df_mut['Variant_Classification'].isin(functional)].copy()
        mut_ev = df_mut_func[['Sample_Id', 'Hugo_Symbol']].drop_duplicates()
        events = pd.concat([events, mut_ev], ignore_index=True)
        print(f"[*] {cohort_name}: Caricate Mutazioni")

    # --- B. LOGICA CNA (Mode 2) ---
    if mode == 2 and os.path.exists(cna_file):
        df_cna = pd.read_csv(cna_file, sep='\t')
        df_cna_long = df_cna.melt(id_vars='Hugo_Symbol', var_name='Sample_Id', value_name='Status')
        df_cna_f = df_cna_long[df_cna_long['Status'].isin([2, -2])].copy()
        cna_ev = df_cna_f[['Sample_Id', 'Hugo_Symbol']].drop_duplicates()
        events = pd.concat([events, cna_ev], ignore_index=True)
        print(f"[*] {cohort_name}: Caricate CNA (Amp/Del)")

    # --- C. LOGICA SV (Mode 3) ---
    if mode == 3 and os.path.exists(sv_file):
        try:
            df_sv = pd.read_csv(sv_file, sep=None, engine='python', on_bad_lines='skip')
            sv1 = df_sv[['Sample_Id', 'Site1_Hugo_Symbol']].rename(columns={'Site1_Hugo_Symbol': 'Hugo_Symbol'})
            sv2 = df_sv[['Sample_Id', 'Site2_Hugo_Symbol']].rename(columns={'Site2_Hugo_Symbol': 'Hugo_Symbol'})
            sv_ev = pd.concat([sv1, sv2], ignore_index=True).dropna(subset=['Hugo_Symbol'])
            sv_ev = sv_ev[sv_ev['Hugo_Symbol'] != 'Unknown'].drop_duplicates()
            events = pd.concat([events, sv_ev], ignore_index=True)
            print(f"[*] {cohort_name}: Caricate SV (Site1+Site2)")
        except Exception as e:
            print(f"[!] Errore SV: {e}")

    # --- ELABORAZIONE MATRICE ---
    events = events.drop_duplicates(subset=['Sample_Id', 'Hugo_Symbol'])
    pat_gene_counts = pd.crosstab(events['Sample_Id'], events['Hugo_Symbol'])
    binary_matrix = (pat_gene_counts > 0).astype(int)
    
    # Includi tutti i pazienti originali dal file mutazioni
    df_base = pd.read_csv(mut_file, sep='\t')
    tutti_i_pazienti = df_base['Sample_Id'].unique() 
    binary_matrix = binary_matrix.reindex(tutti_i_pazienti, fill_value=0)
    
    print(f"\n--- ANALISI [{tag}]: {cohort_name.upper()} ---")
    print(f"[*] Geni unici alterati: {binary_matrix.shape[1]}")
    
    # Salvataggio con TAG nel nome (cosi non sovrascrivi i file se cambi modalita)
    os.makedirs(output_dir, exist_ok=True)
    binary_matrix.to_csv(os.path.join(output_dir, f"M_binary_{cohort_name}_{tag}.csv"), sep='\t')
    
    co_occ_matrix = binary_matrix.T.dot(binary_matrix)
    co_occ_matrix.to_csv(os.path.join(output_dir, f"M_cooccurrence_{cohort_name}_{tag}.tsv"), sep='\t')

    # Heatmap
    top_genes = binary_matrix.sum().sort_values(ascending=False).index[:TOP_GENES_HEATMAP]
    subset_matrix = co_occ_matrix.loc[top_genes, top_genes]
    plt.figure(figsize=(10, 8))
    sns.heatmap(subset_matrix, cmap="YlOrRd", annot=True, fmt='d')
    plt.title(f"Co-occorrenze {tag} - {cohort_name.upper()}")
    print(f"[*] Heatmap {tag} generata per i top {TOP_GENES_HEATMAP} geni.")
    plt.savefig(os.path.join(output_dir, f"Heatmap_{cohort_name}_{tag}.png"))
    plt.close()
    
    print(f"Dati {tag} salvati correttamente.")

# Esecuzione
for name, path in COORTI.items():
    generate_star_cooccurrence_data(path, name, f"{path}/co_occurr_output", mode=ANALYSIS_MODE)

## Analisi statistica, calcolo Stats co-occorrenza Star e Full


In [ ]:
# ==========================================
# CELLA 5: ANALISI STATISTICA (STAR + FULL PATH)
# ==========================================

# 1. CREIAMO I DIZIONARI GLOBALI PER SALVARE I DATAFRAME
dict_star_results = {}
dict_full_results = {}  # Inserito anche questo nel caso ti serva per il plot

def analyze_significance(binary_matrix_file, output_path, target_gene, name):
    """
    Esegue il Test di Fisher per KRAS vs Tutti (Star Analysis).
    Ho aggiunto 'name' come parametro per poter leggere le soglie specifiche del tumore.
    """
    if not os.path.exists(binary_matrix_file): 
        print(f"[!] File non trovato per analisi Star: {binary_matrix_file}")
        return None
        
    df_bin = pd.read_csv(binary_matrix_file, sep='\t', index_col=0).fillna(0)
    
    if target_gene not in df_bin.columns: 
        print(f"[!] {target_gene} non trovato nella matrice.")
        return None

    genes = [g for g in df_bin.columns if g != target_gene]
    results = []

    for gene_x in genes:
        a = ((df_bin[target_gene] == 1) & (df_bin[gene_x] == 1)).sum()
        b = ((df_bin[target_gene] == 1) & (df_bin[gene_x] == 0)).sum()
        c = ((df_bin[target_gene] == 0) & (df_bin[gene_x] == 1)).sum()
        d = ((df_bin[target_gene] == 0) & (df_bin[gene_x] == 0)).sum()

        table = [[a, b], [c, d]]
        odds_ratio, p_value = fisher_exact(table, alternative='greater')
        log2or = np.log2(odds_ratio) if odds_ratio > 0 else -10.0

        results.append({
            'Gene_B': gene_x, 
            'Co_Occurrence_Count': a, 
            'P_Value': p_value, 
            'Log2OR': log2or
        })

    results_df = pd.DataFrame(results)
    results_df['P_Adj'] = multipletests(results_df['P_Value'], method='fdr_bh')[1]
    results_df['Log2OR'] = results_df['Log2OR'].replace([np.inf, -np.inf], [10.0, -10.0])

    results_df.to_csv(output_path, sep='\t', index=False)
    
    # --- PRINTS STATISTICHE STAR ---
    n_tot = len(results_df)
    n_padj = (results_df['P_Adj'] < P_VALUE_THRESH).sum()
    n_log = (results_df['Log2OR'] > LOG2OR_THRESH[name]).sum()
    n_both = ((results_df['P_Adj'] < P_VALUE_THRESH) & (results_df['Log2OR'] > LOG2OR_THRESH[name])).sum()
    
    print(f"✅ Star_Results salvato: {output_path}")
    print(f"   📊 RIEPILOGO {target_gene} vs TUTTI (Totale testati: {n_tot} geni)")
    print(f"      🔹 Con P_Adj < {P_VALUE_THRESH}: {n_padj}")
    print(f"      🔹 Con Log2OR > {LOG2OR_THRESH[name]}: {n_log}")
    print(f"      🔹 Entrambi (Significativi & Forti): {n_both}\n")

    return results_df

# ==========================================
# ESECUZIONE: GENERAZIONE ENTRAMBI I FILE CON STATISTICHE
# ==========================================
labels_tag = {1: "MUT", 2: "CNA", 3: "SV"}
tag = labels_tag.get(ANALYSIS_MODE, "CUSTOM")

for name in COORTI.keys():
    print(f"==================================================")
    print(f"🔬 INIZIO ANALISI COORTE: {name.upper()}")
    print(f"==================================================")
    
    output_dir = f"./kras_{name}/co_occurr_output"
    bin_file = f"{output_dir}/M_binary_{name}_{tag}.csv"
    
    if not os.path.exists(bin_file):
        print(f"❌ File {bin_file} non trovato, salto la coorte.")
        continue

    # --- 1. STAR RESULTS (KRAS vs Tutti) ---
    out_star = f"{output_dir}/Star_Results_{name}.csv"
    # Salva il risultato temporaneo nella variabile df_star (nota: ho aggiunto 'name' come argomento per stampare i log corretti)
    df_star = analyze_significance(bin_file, out_star, TARGET_GENE, name)
    
    # 2. SE IL DF NON E' VUOTO, SALVALO NEL DIZIONARIO USANDO IL NOME DEL TUMORE
    if df_star is not None:
        dict_star_results[name] = df_star

    # --- 3. FULL RESULTS (Tutti vs Tutti con P-Value e Log2OR) ---
    df_bin = pd.read_csv(bin_file, sep='\t', index_col=0).fillna(0)
    n_total = len(df_bin)
    genes = df_bin.columns.tolist()
    gene_counts = df_bin.sum(axis=0).astype(int)
    
    full_results = []
    print(f"[*] Calcolo statistiche Full Path per {name} (coppie potenziali: {len(genes)*(len(genes)-1)//2})...")

    # Iteriamo su tutte le combinazioni possibili di geni
    for g1, g2 in combinations(genes, 2):
        a = ((df_bin[g1] == 1) & (df_bin[g2] == 1)).sum()
        
        # Filtro minimo per velocizzare (almeno 2 pazienti con entrambi i geni mutati)
        if a < 2: continue 
            
        b = gene_counts[g1] - a
        c = gene_counts[g2] - a
        d = n_total - (a + b + c)

        table = [[a, b], [c, d]]
        odds_ratio, p_value = fisher_exact(table, alternative='greater')
        log2or = np.log2(odds_ratio) if odds_ratio > 0 else -10.0

        full_results.append({
            'Gene_A': g1,
            'Gene_B': g2,
            'Co_Occurrence_Count': a,
            'P_Value': p_value,
            'Log2OR': log2or
        })

    if full_results:
        df_full = pd.DataFrame(full_results)
        # Correzione FDR su tutte le coppie testate
        df_full['P_Adj'] = multipletests(df_full['P_Value'], method='fdr_bh')[1]
        df_full['Log2OR'] = df_full['Log2OR'].replace([np.inf, -np.inf], [10.0, -10.0])
        
        out_full = f"{output_dir}/Full_Results_{name}.csv"
        df_full.to_csv(out_full, sep='\t', index=False)
        
        # 4. SALVIAMO ANCHE IL FULL RESULTS NEL SUO DIZIONARIO
        dict_full_results[name] = df_full
        
        # --- PRINTS STATISTICHE FULL PATH ---
        n_tot_coppie = len(df_full)
        n_padj = (df_full['P_Adj'] < P_VALUE_THRESH).sum()
        n_log = (df_full['Log2OR'] > LOG2OR_THRESH[name]).sum()
        n_both = ((df_full['P_Adj'] < P_VALUE_THRESH) & (df_full['Log2OR'] > LOG2OR_THRESH[name])).sum()
        
        n_strict = ((df_full['P_Adj'] < P_VALUE_THRESH) & 
                    (df_full['Log2OR'] > LOG2OR_THRESH[name]) & 
                    (df_full['Co_Occurrence_Count'] >= MIN_COOCCURRENCE_NETWORK[name])).sum()

        print(f"✅ Full_Results salvato: {out_full}")
        print(f"   📊 RIEPILOGO FULL PATH (Coppie con co-occorrenza >= 2: {n_tot_coppie})")
        print(f"      🔹 Con P_Adj < {P_VALUE_THRESH}: {n_padj}")
        print(f"      🔹 Con Log2OR > {LOG2OR_THRESH[name]}: {n_log}")
        print(f"      🔹 Entrambi (P_Adj e Log2OR): {n_both}")
        print(f"      ⭐ Archi candidati per la rete (Entrambi + Co-Occorrenze >= {MIN_COOCCURRENCE_NETWORK[name]}): {n_strict}\n")
    else:
        print(f"[!] Nessuna coppia con co-occorrenza >= 2 trovata per {name}\n")

## Mutual exclusivity


In [ ]:
# ==========================================
# NUOVA CELLA: CALCOLO MUTUA ESCLUSIVITÀ
# ==========================================

import pandas as pd
import numpy as np
from scipy.stats import fisher_exact
import os

def calculate_mutual_exclusivity(binary_matrix_file, output_dir, cohort_name, target_gene='KRAS'):
    print(f"\n--- 🚫 CALCOLO MUTUA ESCLUSIVITÀ: {cohort_name.upper()} ---")
    
    if not os.path.exists(binary_matrix_file):
        print(f"[!] File matrice binaria mancante per {cohort_name}. Salto.")
        return None
        
    # Carica la matrice binaria pazienti-geni (dove 1 = mutato, 0 = wild-type)
    df_bin = pd.read_csv(binary_matrix_file, sep='\t', index_col=0)
    
    if target_gene not in df_bin.columns:
        print(f"[!] {target_gene} non trovato nella matrice.")
        return None
        
    target_mut = df_bin[target_gene]
    results = []
    
    for gene in df_bin.columns:
        if gene == target_gene:
            continue
            
        gene_mut = df_bin[gene]
        
        # Costruzione della Tabella di Contingenza
        # [[Entrambi mutati,      Solo KRAS mutato],
        #  [Solo Gene B mutato,   Nessuno dei due mutato]]
        both_mut = len(df_bin[(target_mut == 1) & (gene_mut == 1)])
        only_target = len(df_bin[(target_mut == 1) & (gene_mut == 0)])
        only_gene = len(df_bin[(target_mut == 0) & (gene_mut == 1)])
        neither_mut = len(df_bin[(target_mut == 0) & (gene_mut == 0)])
        
        table = [[both_mut, only_target],
                 [only_gene, neither_mut]]
                 
        # TEST DI FISHER - Notare l'uso di alternative='less'
        oddsratio, p_value = fisher_exact(table, alternative='less')
        
        # Calcolo Log2OR gestendo i casi limite matematici (divisione per zero)
        if oddsratio == 0:
            log2or = -np.inf # Mutua esclusività perfetta (non mutano MAI insieme)
        elif oddsratio == np.inf:
            log2or = np.inf
        else:
            log2or = np.log2(oddsratio)
            
        results.append({
            'Gene_A': target_gene,
            'Gene_B': gene,
            'Co_Occurrence_Count': both_mut,
            'Only_KRAS_Count': only_target,
            'Only_Gene_B_Count': only_gene,
            'Neither_Count': neither_mut,
            'P_Value': p_value,
            'Log2OR': log2or
        })
        
    res_df = pd.DataFrame(results)
    
    # Applichiamo i filtri: p-value significativo e Log2OR fortemente negativo
    sig_me_df = res_df[(res_df['P_Value'] <= P_VALUE_THRESH_ME) & 
                       (res_df['Log2OR'] <= LOG2OR_THRESH_ME)]
                       
    # Salvataggio su file
    os.makedirs(output_dir, exist_ok=True)
    out_file = os.path.join(output_dir, f"Mutual_Exclusivity_Results_{cohort_name}.csv")
    
    # Ordiniamo per P-value crescente (i più significativi in alto)
    sig_me_df = sig_me_df.sort_values('P_Value')
    sig_me_df.to_csv(out_file, sep='\t', index=False)
    
    print(f"[*] Analizzati {len(df_bin.columns)-1} geni rispetto a {target_gene}.")
    print(f"[*] Trovati {len(sig_me_df)} geni significativamente MUTUALMENTE ESCLUSIVI.")
    print(f"✅ Risultati salvati in: {out_file}")
    
    return sig_me_df

# ==========================================
# ESECUZIONE PER TUTTE LE COORTI
# ==========================================
labels = {1: "MUT", 2: "CNA", 3: "SV", 4: "INTEGRATED"}
tag = labels.get(ANALYSIS_MODE, "CUSTOM")

dfs_esclusivita = {}
for name in COORTI.keys():
    # Assicurati che il percorso della matrice binaria sia corretto in base a dove la salvi
    dfs_esclusivita[name] = calculate_mutual_exclusivity(
        binary_matrix_file=f"./kras_{name}/co_occurr_output/M_binary_{name}_{tag}.csv",
        output_dir=f"./kras_{name}/co_occurr_output/",
        cohort_name=name,
        target_gene='KRAS'
    )

## Plot


In [ ]:
# ==========================================
# CELLA 6: VOLCANO PLOT (VERSIONE P-VALUE GREZZO)
# ==========================================
def plot_volcano(df_results, cohort_name, log2or_thresh, p_thresh):
    """
    Genera un Volcano Plot basato sul P-Value grezzo.
    Asse X: Log2 Odds Ratio (Forza dell'associazione)
    Asse Y: -Log10 P-Value (Significatività statistica)
    """
    if df_results is None or df_results.empty:
        print(f"[!] Dati insufficienti per il Volcano Plot di {cohort_name}")
        return
    
    df_plot = df_results.copy()
    
    # Trasformazione Logaritmica dell'asse Y (P-Value grezzo)
    # Usiamo 1e-300 per evitare errori matematici con p-value pari a 0
    df_plot['neg_log10_p'] = -np.log10(df_plot['P_Value'] + 1e-300)
    
    # Definizione dello stato in base alle variabili globali
    cond_sig = (df_plot['Log2OR'] >= log2or_thresh) & (df_plot['P_Value'] <= p_thresh)
    df_plot['Stato'] = np.where(cond_sig, 'Significativo', 'Non Significativo')
    
    # Setup del grafico
    plt.figure(figsize=(10, 7))
    
    # Plot dei punti
    scatter = sns.scatterplot(
        data=df_plot, 
        x='Log2OR', 
        y='neg_log10_p', 
        hue='Stato',
        palette={'Significativo': '#d62728', 'Non Significativo': '#b0b0b0'},
        alpha=0.6,
        s=70,
        edgecolor='w'
    )
    
    # Linee tratteggiate per le soglie impostate nella Cella 0
    plt.axvline(x=log2or_thresh, color='red', linestyle='--', alpha=0.5, label=f'Soglia Log2OR > {log2or_thresh}')
    plt.axhline(y=-np.log10(p_thresh), color='blue', linestyle='--', alpha=0.5, label=f'Soglia P-Value < {p_thresh}')
    
    # Annotazione dei geni significativi (solo quelli che superano entrambe le soglie)
    sig_list = df_plot[cond_sig]
    for _, row in sig_list.iterrows():
        plt.text(
            row['Log2OR'] + 0.05, 
            row['neg_log10_p'] + 0.1, 
            row['Gene_B'], 
            fontsize=9, 
            weight='bold',
            alpha=0.8
        )
        
    # Personalizzazione etichette e titoli
    plt.title(f'Volcano Plot Co-occorrenze: {cohort_name.upper()} (Senza correzione FDR)', fontsize=14)
    plt.xlabel('Forza dell\'associazione ($Log_2$ Odds Ratio)', fontsize=12)
    plt.ylabel('Significatività ($-Log_{10}$ P-Value Grezzo)', fontsize=12)
    
    # Posizionamento legenda
    plt.legend(loc='upper left', bbox_to_anchor=(1, 1))
    
    sns.despine()
    plt.tight_layout()
    plt.show()

    # LOG DI DETTAGLIO
    print(f"\n--- 📈 VISUALIZZAZIONE: {cohort_name.upper()} ---")
    print(f"[*] Geni totali mappati: {len(df_plot)}")
    print(f"[*] Geni sopra la soglia di significatività: {cond_sig.sum()}")

# ==========================================
# ESECUZIONE
# ==========================================
for name in COORTI.keys():
    # Utilizza le variabili globali LOG2OR_THRESH e P_VALUE_THRESH definite nella Cella 0
    plot_volcano(dict_star_results.get(name), name, log2or_thresh=LOG2OR_THRESH[name], p_thresh=P_VALUE_THRESH)

## Network (star)


In [ ]:
# ==========================================
# CELLA 7: NETWORK INTERATTIVA (FIX OVERFLOW)
# ==========================================
def build_interactive_star_network(stat_file, mat_file, output_dir, cohort_name, target_gene=TARGET_GENE):
    if not os.path.exists(stat_file) or not os.path.exists(mat_file):
        print(f"[!] File necessari non trovati per {cohort_name}.")
        return None

    # 1. Carica i risultati e filtra
    df_stats = pd.read_csv(stat_file, sep='\t')
    df_sig = df_stats[(df_stats['Log2OR'] >= LOG2OR_THRESH[cohort_name]) & (df_stats['P_Value'] <= P_VALUE_THRESH)].copy()

    if df_sig.empty:
        print(f"[-] {cohort_name.upper()}: Nessun gene supera le soglie per la rete.")
        return None

    coocc_matrix = pd.read_csv(mat_file, sep='\t', index_col=0)
    G = nx.Graph()
    G.add_node(target_gene, size=40, color='#d62728', title=f"TARGET: {target_gene}", font={'size': 20})

    # 4. Aggiungi i nodi e gli archi verso il target
    for _, row in df_sig.iterrows():
        gene_b = row['Gene_B']
        
        # --- FIX PROTEZIONE INFINITO ---
        log_val = row['Log2OR']
        # Se Log2OR è infinito, lo blocchiamo a un valore massimo (es. 10) per il calcolo grafico
        if np.isinf(log_val):
            log_val = 10.0 
        
        node_size = float(15 + (log_val * 8))
        # -------------------------------

        weight = int(row['Co_Occurrence_Count'])
        p_val = float(row['P_Value'])
        
        tooltip = f"Gene: {gene_b}<br>Log2OR: {row['Log2OR']:.2f}<br>P-Value: {p_val:.4e}<br>Co-occorrenze: {weight}"
        
        G.add_node(gene_b, size=node_size, color='#1f77b4', title=tooltip)
        G.add_edge(target_gene, gene_b, weight=weight, value=weight, color='#999999')

        # 5. Archi secondari tra i partner
        for other_gene in df_sig['Gene_B']:
            if gene_b != other_gene and gene_b in coocc_matrix.index and other_gene in coocc_matrix.columns:
                edge_weight = int(coocc_matrix.loc[gene_b, other_gene])
                if edge_weight > MIN_COOCCURRENCE_NETWORK[cohort_name]:  # Solo se superano la soglia di co-occorrenza
                    G.add_edge(gene_b, other_gene, weight=edge_weight, value=edge_weight, color='#e0e0e0', alpha=0.5)

    # 6. Configurazione PyVis
    net = Network(notebook=True, cdn_resources='remote', height="700px", width="100%", bgcolor="#ffffff")
    net.from_nx(G)
    net.repulsion(node_distance=200)

    os.makedirs(output_dir, exist_ok=True)
    html_path = os.path.join(output_dir, f"Network_{cohort_name}.html")
    net.show(html_path)
    
    print(f"\n--- 🌐 NETWORK GENERATA: {cohort_name.upper()} ---")
    print(f"[*] Nodi totali: {G.number_of_nodes()}")
    print(f"✅ Rete salvata in: {html_path}")
    print(f"[->]Numero di archi: {G.number_of_edges()}")
    return G

# Esecuzione
for name in COORTI.keys():
    build_interactive_star_network(
        f"./kras_{name}/co_occurr_output/Star_Results_{name}.csv",
        f"./kras_{name}/co_occurr_output/M_cooccurrence_gene_gene_{name}.tsv",
        f"./kras_{name}/networks/",
        name
    )

## Metriche network star


In [ ]:
# ==========================================
# CELLA 7: CALCOLO METRICHE DI RETE E IDENTIFICAZIONE HUB
# ==========================================

def calculate_network_metrics(stat_file, coocc_matrix_file, output_dir, cohort_name):
    print(f"\n--- 📊 CALCOLO METRICHE DI RETE: {cohort_name.upper()} ---")
    
    if not os.path.exists(stat_file) or not os.path.exists(coocc_matrix_file):
        print(f"[!] File mancanti per {cohort_name}. Salto.")
        return None
        
    # 1. Lettura file risultati e RI-APPLICAZIONE FILTRI STATISTICI
    stat_df = pd.read_csv(stat_file, sep='\t')
    
    # Filtriamo per trovare i partner validi come fatto per il Plot
    sig_genes_df = stat_df[(stat_df['P_Value'] <= P_VALUE_THRESH) & 
                           (stat_df['Log2OR'] >= LOG2OR_THRESH[cohort_name])]
    
    valid_genes = sig_genes_df['Gene_B'].tolist()
    if TARGET_GENE not in valid_genes:
        valid_genes.append(TARGET_GENE)
        
    # 2. Caricamento della matrice di co-occorrenza gene-gene globale
    coocc_df = pd.read_csv(coocc_matrix_file, sep='\t', index_col=0)
    
    # 3. Costruzione del Grafo (NetworkX)
    G = nx.Graph()
    
    # Aggiunta Nodi validati
    for gene in valid_genes:
        if gene in coocc_df.columns:
            G.add_node(gene)
            
    # Aggiunta Archi
    for i in range(len(valid_genes)):
        for j in range(i + 1, len(valid_genes)):
            g1 = valid_genes[i]
            g2 = valid_genes[j]
            
            if g1 in coocc_df.columns and g2 in coocc_df.columns:
                weight = coocc_df.loc[g1, g2]
                
                # Crea l'arco se supera il filtro MIN_COOCCURRENCE_NETWORK 
                # o se è direttamente connesso al TARGET_GENE (KRAS)
                if weight >= MIN_COOCCURRENCE_NETWORK[cohort_name] or (g1 == TARGET_GENE or g2 == TARGET_GENE):
                    G.add_edge(g1, g2, weight=weight)
                    
    # 4. Calcolo Metriche Topologiche
    if G.number_of_nodes() == 0:
        print("[!] Nessun nodo nella rete per calcolare le metriche.")
        return None
        
    degree_cent = nx.degree_centrality(G)
    betweenness_cent = nx.betweenness_centrality(G)
    try:
        eigenvector_cent = nx.eigenvector_centrality(G, max_iter=1000)
    except:
        eigenvector_cent = nx.degree_centrality(G) # Fallback in caso di mancata convergenza

    # Nuove metriche da calcolare prima del ciclo 'for node in G.nodes():'
    clustering_coeffs = nx.clustering(G)
    closeness_cent = nx.closeness_centrality(G)
    
    metrics_data = []
    for node in G.nodes():
        metrics_data.append({
            'Gene': node,
            'Degree': G.degree(node),
            'Degree_Centrality': round(degree_cent.get(node, 0), 4),
            'Betweenness_Centrality': round(betweenness_cent.get(node, 0), 4),
            'Eigenvector_Centrality': round(eigenvector_cent.get(node, 0), 4),
            'Closeness_Centrality': round(closeness_cent.get(node, 0), 4), # NUOVO
            'Clustering_Coefficient': round(clustering_coeffs.get(node, 0), 4) # NUOVO
        })
        
    metrics_df = pd.DataFrame(metrics_data)
    # Ordiniamo i risultati in base al grado di connessione per trovare subito gli HUB
    metrics_df = metrics_df.sort_values(by='Degree_Centrality', ascending=False)
    
    # 5. Salvataggio su file CSV
    os.makedirs(output_dir, exist_ok=True)
    out_path = os.path.join(output_dir, f"Network_Metrics_{cohort_name}.csv")
    metrics_df.to_csv(out_path, sep='\t', index=False)
    
    print(f"[*] Nodi: {G.number_of_nodes()} | Archi: {G.number_of_edges()}")
    if not metrics_df.empty:
        print(f"[*] Top 3 Hub identificati: {metrics_df['Gene'].head(3).tolist()}")
    print(f"✅ Metriche salvate in: {out_path}")
    
    return metrics_df

# ==========================================
# ESECUZIONE PER TUTTE LE COORTI
# ==========================================
dfs_metriche = {}
for name in COORTI.keys():
    dfs_metriche[name] = calculate_network_metrics(
        stat_file=f"./kras_{name}/co_occurr_output/Star_Results_{name}.csv",
        coocc_matrix_file=f"./kras_{name}/co_occurr_output/M_cooccurrence_gene_gene_{name}.tsv",
        output_dir=f"./kras_{name}/networks/",
        cohort_name=name
    )

## FULL PATH


### Grid search per vedere numero archi e geni a seconda dei parametri in un file


In [ ]:
# ==========================================
# 1. DEFINIZIONE GRIGLIA DI PARAMETRI ESTESA
# ==========================================
min_cooccurrence_values = [5, 10, 15, 20, 25, 30, 40, 50]
log2or_values = [1.0, 1.5, 2.0, 2.5, 3.0, 4.0, 5.0]

print("Inizio estrazione metriche Grid Search per ogni coorte...")

# ==========================================
# 2. CICLO PRINCIPALE SULLE COORTI
# ==========================================
for name in COORTI.keys():
    print(f"\n---> Elaborazione coorte: {name.upper()}")
    
    file_path = f"./kras_{name}/co_occurr_output/Full_Results_{name}.csv"
    
    if not os.path.exists(file_path):
        print(f"ATTENZIONE: File {file_path} non trovato. Salto la coorte {name}.")
        continue
        
    df = pd.read_csv(file_path, sep='\t')

    # Lista dove salveremo i risultati di ogni parametrizzazione
    grid_results = []

    # --- 2.3 Iterazione Grid Search ---
    for min_cooc in min_cooccurrence_values:
        for log2or in log2or_values:
            
            # 1. Filtro
            filtered_df = df[(df['Co_Occurrence_Count'] >= min_cooc) & (df['Log2OR'] >= log2or)]
            
            # 2. Creazione Grafo
            G = nx.from_pandas_edgelist(filtered_df, 'Gene_A', 'Gene_B', ['Co_Occurrence_Count', 'Log2OR'])
            G.remove_nodes_from(list(nx.isolates(G)))
            
            # 3. Estrazione Metriche Base
            num_nodes = G.number_of_nodes()
            num_edges = G.number_of_edges()
            
            num_clusters = 0
            density = 0.0
            
            # 4. Estrazione Metriche Avanzate (solo se la rete non è vuota)
            if num_edges > 0:
                density = nx.density(G)
                try:
                    communities = nx.community.louvain_communities(G, weight='Co_Occurrence_Count')
                    num_clusters = len(communities)
                except AttributeError:
                    try:
                        communities = nx.community.greedy_modularity_communities(G, weight='Co_Occurrence_Count')
                        num_clusters = len(communities)
                    except:
                        num_clusters = 0
            
            # 5. Salvataggio della riga
            grid_results.append({
                'MIN_COOCCURRENCE': min_cooc,
                'LOG2OR_THRESH': log2or,
                'Num_Nodes': num_nodes,
                'Num_Edges': num_edges,
                'Num_Clusters': num_clusters,
                'Network_Density': round(density, 4)
            })

    # --- 2.4 Creazione e Salvataggio del DataFrame ---
    results_df = pd.DataFrame(grid_results)
    
    # Crea cartella se non esiste
    output_dir = f"./kras_{name}/networks"
    os.makedirs(output_dir, exist_ok=True)
    
    # Salva in formato CSV (o Excel se preferisci)
    output_file = f"{output_dir}/grid_search_metrics_{name}.csv"
    results_df.to_csv(output_file, sep='\t', index=False)
    
    print(f"✅ Risultati salvati con successo in '{output_file}'.")
    # Facoltativo: stampa le prime 5 righe per un controllo rapido a schermo
    print(results_df.head())

print("\nTutte le metriche della Grid Search sono state estratte!")

### Grid search per vedere le varie reti a seconda dei parametri


In [ ]:
# ==========================================
# 1. DEFINIZIONE GRIGLIA DI PARAMETRI ESTESA
# ==========================================
# Griglia più ampia per tumori ad alto carico mutazionale
min_cooccurrence_values = [5, 10, 15, 20, 25, 30, 40, 50]
log2or_values = [0.0, 1.0, 1.5, 2.0, 2.5, 3.0, 4.0]

num_rows = len(min_cooccurrence_values)
num_cols = len(log2or_values)

print("Inizio generazione dei collage di reti per ogni coorte (Griglia Estesa)...")

# ==========================================
# 2. CICLO PRINCIPALE SULLE COORTI
# ==========================================
for name in COORTI.keys():
    print(f"\n---> Elaborazione coorte: {name.upper()}")
    
    # --- 2.1 Caricamento Dati ---
    file_path = f"./kras_{name}/co_occurr_output/Full_Results_{name}.csv"
    
    if not os.path.exists(file_path):
        print(f"ATTENZIONE: File {file_path} non trovato. Assicurati che il percorso e il nome siano corretti. Salto la coorte {name}.")
        continue
        
    df = pd.read_csv(file_path, sep='\t')

    # --- 2.2 Impostazione del Grafico (Collage) ---
    # Tela ingrandita (50x45) per accomodare la nuova griglia 8x7
    fig, axes = plt.subplots(nrows=num_rows, ncols=num_cols, figsize=(50, 45))
    plt.subplots_adjust(wspace=0.3, hspace=0.3)

    # --- 2.3 Iterazione Grid Search ---
    for row_idx, min_cooc in enumerate(min_cooccurrence_values):
        for col_idx, log2or in enumerate(log2or_values):
            ax = axes[row_idx, col_idx]
            
            # Filtro
            filtered_df = df[(df['Co_Occurrence_Count'] >= min_cooc) & (df['Log2OR'] >= log2or)]
            
            # Creazione Grafo
            G = nx.from_pandas_edgelist(filtered_df, 'Gene_A', 'Gene_B', ['Co_Occurrence_Count', 'Log2OR'])
            G.remove_nodes_from(list(nx.isolates(G)))
            
            num_nodes = G.number_of_nodes()
            num_edges = G.number_of_edges()
            
            title_text = f"MinCooc >= {min_cooc}\nLog2OR >= {log2or}\n(Nodes: {num_nodes}, Edges: {num_edges})"
            
            # Se la rete è vuota o troppo sparsa
            if num_edges == 0:
                if num_nodes > 0:
                    nx.draw_networkx_nodes(G, nx.spring_layout(G, seed=42), node_size=50, node_color='lightgray', ax=ax)
                    ax.set_title(title_text + "\n(Disconnected Nodes)", fontsize=12, fontweight='bold')
                else:
                    ax.text(0.5, 0.5, "Rete Vuota", horizontalalignment='center', verticalalignment='center', fontsize=14, transform=ax.transAxes)
                    ax.set_title(title_text, fontsize=12, fontweight='bold')
                ax.axis('off')
                continue
            
            # Calcolo Cluster
            try:
                communities = nx.community.louvain_communities(G, weight='Co_Occurrence_Count')
            except AttributeError:
                communities = nx.community.greedy_modularity_communities(G, weight='Co_Occurrence_Count')
            
            color_map = {}
            for i, comm in enumerate(communities):
                for node in comm:
                    color_map[node] = i
            colors = [color_map.get(node, 0) for node in G.nodes()]

            # Disegno Rete
            pos = nx.spring_layout(G, k=0.8, seed=42) # Aumentato 'k' per spingere i nodi un po' più lontani se la rete è densa
            
            # Adattamento nodi in base a reti potenzialmente molto più grandi (Colon/Lung)
            if num_nodes > 200:
                node_sz, label_sz = 50, 3
            elif num_nodes > 100:
                node_sz, label_sz = 100, 5
            elif num_nodes > 50:
                node_sz, label_sz = 200, 7
            else:
                node_sz, label_sz = 350, 10
                
            nx.draw_networkx_nodes(G, pos, node_size=node_sz, node_color=colors, cmap=plt.cm.Set3, alpha=0.9, ax=ax)
            weights = [G[u][v]['Co_Occurrence_Count'] / 5.0 for u, v in G.edges()] # Scalato maggiormente il peso per via degli alti Count
            nx.draw_networkx_edges(G, pos, width=weights, edge_color='gray', alpha=0.3, ax=ax)
            nx.draw_networkx_labels(G, pos, font_size=label_sz, font_family='sans-serif', ax=ax)
            
            ax.set_title(title_text, fontsize=12, fontweight='bold')
            ax.axis('off')

    # --- 2.4 Salvataggio Finale ---
    plt.suptitle(f"Coorte {name.upper()}: Grid Search Estesa dei Parametri\n(Righe: MIN_COOCCURRENCE, Colonne: LOG2OR_THRESH)", fontsize=32, fontweight='bold')
    
    # Crea cartella se non esiste
    output_dir = f"./kras_{name}/networks"
    os.makedirs(output_dir, exist_ok=True)
    
    output_file = f"{output_dir}/grid_search_networks_collage_extended.png"
    plt.savefig(output_file, dpi=120) # DPI leggermente ridotto per non creare file da 100MB, vista la griglia enorme
    print(f"Collage salvato con successo come '{output_file}'.")
    
    # Libera la RAM
    plt.close(fig)

print("\nTutte le analisi estese sono state completate!")

In [ ]:
# ROBA LETTA NEL CODICE DELL'ALTRO GRUPPO
# top_n_genes = 45
# min_shared = 40
# max_edges = 220
# edges_filtered = edges[edges['log2_OR'] > 3].copy()
# edges_sig = edges_fisher[(edges_fisher["odds_ratio"] > 1) & (edges_fisher["p_adj"] < 0.05)].copy()
